# Data Leakage Check — Train/Val Similarity Analysis

DeepGlobe tiles are cut from larger satellite images. Adjacent tiles may share roads, buildings, and geographic context. If our random train/val split places adjacent tiles in different sets, the model effectively sees part of the validation data during training — inflating metrics.

### Strategy
Instead of comparing all 5292x934 pairs, we focus on the **highest IoU samples** in both sets — if leakage exists, it would inflate these scores the most.

### Methods compared
1. **CLIP embeddings** — semantic similarity (understands scene content)
2. **ResNet encoder features** — visual similarity from our trained model

### Caching
Every expensive step (IoU computation, embedding extraction, similarity matrices) is cached to `data/interim/leakage_cache/`. If a cell fails downstream, re-running skips the expensive parts.

---
## 1. Setup

In [51]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "open-clip-torch"], check=True)
    PROJECT_ROOT = __import__('pathlib').Path(REPO_DIR).resolve()
else:
    PROJECT_ROOT = __import__('pathlib').Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from PIL import Image

plt.style.use("ggplot")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Cache directory ---
CACHE_DIR = Path("data/interim/leakage_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Cache: {CACHE_DIR}")

Device: cpu
Cache: data/interim/leakage_cache


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/road_segmentation")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR = str(DRIVE_DIR / "checkpoints")
    LOG_DIR = str(DRIVE_DIR / "logs")
    print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = "checkpoints"
    LOG_DIR = "logs"
    print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

train_dir = DEEPGLOBE_DATASET_DIR / "train"
if not (train_dir.exists() and any(train_dir.glob("*_sat.*"))):
    print("Downloading dataset...")
    from road_segmentation.data.download import download_dataset
    download_dataset()

sat_count = len(list(train_dir.glob("*_sat.*")))
print(f"Training: {sat_count} image-mask pairs")

---
## 2. Load Train/Val Split and Select Top-IoU Samples

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
train_pairs, val_pairs = split_pairs(pairs, val_ratio=0.15, seed=42)

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

# ====== SET YOUR CHECKPOINT (optional — for IoU-based selection) ======
CHECKPOINT = "/content/drive/MyDrive/road_segmentation/checkpoints/UnetPlusPlus_efficientnet-b4_20260406_231115/best.pth"  # Set to best.pth path, or leave empty for random selection

TOP_K = 500  # Number of top samples from each set to compare

In [ ]:
TOP_K = 500

In [ ]:
import pickle

selection_cache = CACHE_DIR / "top_k_selection.pkl"

if selection_cache.exists():
    print(f"Loading cached selection from {selection_cache}")
    with open(selection_cache, "rb") as f:
        cached = pickle.load(f)
    train_top_idx = cached["train_top_idx"]
    val_top_idx = cached["val_top_idx"]

elif CHECKPOINT:
    print("Computing IoU for all samples (this takes a few minutes)...")
    from road_segmentation.models.factory import create_model
    from road_segmentation.data.transforms import get_val_transform
    from road_segmentation.data.dataset import RoadSegmentationDataset
    from torch.utils.data import DataLoader

    state = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    cfg = state.get("config", {})
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    norm_cfg = cfg.get("normalization", {})
    image_size = data_cfg.get("image_size", 1024)
    mean = norm_cfg.get("mean", [0.485, 0.456, 0.406])
    std = norm_cfg.get("std", [0.229, 0.224, 0.225])

    seg_model = create_model(
        arch=model_cfg.get("arch", "Unet"),
        encoder_name=model_cfg.get("encoder_name", "resnet34"),
        encoder_weights=None,
    )
    seg_model.load_state_dict(state["model_state_dict"])
    seg_model = seg_model.to(device).eval()

    val_transform = get_val_transform(image_size, mean, std)

    def compute_ious(pairs_list, model, transform):
        ds = RoadSegmentationDataset(pairs_list, transform=transform)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
        ious = []
        with torch.no_grad():
            for batch in tqdm(loader, desc="Computing IoU"):
                imgs = batch["image"].to(device)
                masks = batch["mask"]
                probs = torch.sigmoid(model(imgs)).cpu()
                for k in range(len(probs)):
                    pred = (probs[k, 0] >= 0.5).numpy()
                    gt = (masks[k, 0].numpy() > 0)
                    tp = (pred & gt).sum()
                    union = (pred | gt).sum()
                    ious.append(tp / max(union, 1))
        return np.array(ious)

    train_ious = compute_ious(train_pairs, seg_model, val_transform)
    val_ious = compute_ious(val_pairs, seg_model, val_transform)

    train_top_idx = np.argsort(train_ious)[-TOP_K:]
    val_top_idx = np.argsort(val_ious)[-TOP_K:]

    del seg_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Cache
    with open(selection_cache, "wb") as f:
        pickle.dump({"train_top_idx": train_top_idx, "val_top_idx": val_top_idx,
                     "train_ious": train_ious, "val_ious": val_ious}, f)
    print(f"Cached to {selection_cache}")

else:
    print(f"No checkpoint — using random {TOP_K} samples from each set.")
    rng = np.random.RandomState(42)
    train_top_idx = rng.choice(len(train_pairs), TOP_K, replace=False)
    val_top_idx = rng.choice(len(val_pairs), TOP_K, replace=False)

    with open(selection_cache, "wb") as f:
        pickle.dump({"train_top_idx": train_top_idx, "val_top_idx": val_top_idx}, f)

train_selected = [train_pairs[i] for i in train_top_idx]
val_selected = [val_pairs[i] for i in val_top_idx]
print(f"Comparing {TOP_K} train x {TOP_K} val = {TOP_K**2:,} pairs")

---
## 3. Method 1 — ResNet Feature Similarity

In [ ]:
import torchvision.models as models
import torchvision.transforms as T

resnet_cache = CACHE_DIR / "resnet_embeddings.npz"

if resnet_cache.exists():
    print(f"Loading cached ResNet embeddings from {resnet_cache}")
    data = np.load(resnet_cache)
    train_resnet_emb = torch.from_numpy(data["train"])
    val_resnet_emb = torch.from_numpy(data["val"])
else:
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    resnet.fc = torch.nn.Identity()
    resnet = resnet.to(device).eval()

    resnet_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def extract_resnet_embeddings(pairs_list, model, transform):
        embeddings = []
        with torch.no_grad():
            for pair in tqdm(pairs_list, desc="ResNet embeddings"):
                img = Image.open(pair.image_path).convert("RGB")
                tensor = transform(img).unsqueeze(0).to(device)
                feat = model(tensor)
                feat = F.normalize(feat, dim=1)
                embeddings.append(feat.cpu())
        return torch.cat(embeddings, dim=0)

    print("Extracting ResNet embeddings...")
    train_resnet_emb = extract_resnet_embeddings(train_selected, resnet, resnet_transform)
    val_resnet_emb = extract_resnet_embeddings(val_selected, resnet, resnet_transform)

    # Cache
    np.savez(resnet_cache, train=train_resnet_emb.numpy(), val=val_resnet_emb.numpy())
    print(f"Cached to {resnet_cache}")

    del resnet
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Compute similarity matrix
resnet_sim = torch.mm(val_resnet_emb, train_resnet_emb.T).numpy()
print(f"ResNet sim matrix: {resnet_sim.shape} | max={resnet_sim.max():.4f} | mean={resnet_sim.mean():.4f}")

---
## 4. Method 2 — CLIP Embedding Similarity

In [ ]:
clip_cache = CACHE_DIR / "clip_embeddings.npz"

try:
    import open_clip
    CLIP_AVAILABLE = True
except ImportError:
    CLIP_AVAILABLE = False
    print("open-clip-torch not installed. Skipping CLIP.")

if CLIP_AVAILABLE:
    if clip_cache.exists():
        print(f"Loading cached CLIP embeddings from {clip_cache}")
        data = np.load(clip_cache)
        train_clip_emb = torch.from_numpy(data["train"])
        val_clip_emb = torch.from_numpy(data["val"])
    else:
        clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )
        clip_model = clip_model.to(device).eval()

        def extract_clip_embeddings(pairs_list, model, preprocess):
            embeddings = []
            with torch.no_grad():
                for pair in tqdm(pairs_list, desc="CLIP embeddings"):
                    img = Image.open(pair.image_path).convert("RGB")
                    tensor = preprocess(img).unsqueeze(0).to(device)
                    feat = model.encode_image(tensor)
                    feat = F.normalize(feat, dim=1)
                    embeddings.append(feat.cpu())
            return torch.cat(embeddings, dim=0).float()

        print("Extracting CLIP embeddings...")
        train_clip_emb = extract_clip_embeddings(train_selected, clip_model, clip_preprocess)
        val_clip_emb = extract_clip_embeddings(val_selected, clip_model, clip_preprocess)

        # Cache
        np.savez(clip_cache, train=train_clip_emb.numpy(), val=val_clip_emb.numpy())
        print(f"Cached to {clip_cache}")

        del clip_model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    clip_sim = torch.mm(val_clip_emb, train_clip_emb.T).numpy()
    print(f"CLIP sim matrix: {clip_sim.shape} | max={clip_sim.max():.4f} | mean={clip_sim.mean():.4f}")

### Cache similarity matrices

In [ ]:
# Save similarity matrices so downstream cells never need to recompute
sim_cache = CACHE_DIR / "similarity_matrices.npz"
save_dict = {"resnet_sim": resnet_sim}
if CLIP_AVAILABLE:
    save_dict["clip_sim"] = clip_sim
np.savez(sim_cache, **save_dict)
print(f"Similarity matrices cached to {sim_cache}")

# Also save sample IDs for reference
import pickle
ids_cache = CACHE_DIR / "selected_sample_ids.pkl"
with open(ids_cache, "wb") as f:
    pickle.dump({
        "train_ids": [p.sample_id for p in train_selected],
        "val_ids": [p.sample_id for p in val_selected],
        "train_paths": [str(p.image_path) for p in train_selected],
        "val_paths": [str(p.image_path) for p in val_selected],
    }, f)
print(f"Sample IDs cached to {ids_cache}")

print(f"\nAll cached files:")
for p in sorted(CACHE_DIR.glob("*")):
    print(f"  {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

---
## 5. Analysis — Similarity Distributions

Everything below uses only the cached matrices — safe to re-run without recomputing embeddings.

In [52]:
# Reload from cache if needed (e.g., after restart)
if 'resnet_sim' not in dir():
    data = np.load(CACHE_DIR / "similarity_matrices.npz")
    resnet_sim = data["resnet_sim"]
    CLIP_AVAILABLE = "clip_sim" in data.files
    if CLIP_AVAILABLE:
        clip_sim = data["clip_sim"]
    print("Loaded similarity matrices from cache.")

n_plots = 2 if CLIP_AVAILABLE else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

resnet_flat = resnet_sim.ravel()
axes[0].hist(resnet_flat, bins=100, color="#2b8cbe", edgecolor="white", alpha=0.8)
axes[0].axvline(0.95, color="red", linestyle="--", label="Suspicious (>0.95)")
axes[0].axvline(0.90, color="orange", linestyle="--", label="High (>0.90)")
axes[0].set_title(f"ResNet Cosine Similarity (val vs train)\nmax={resnet_flat.max():.4f}")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Count")
axes[0].legend()

if CLIP_AVAILABLE:
    clip_flat = clip_sim.ravel()
    axes[1].hist(clip_flat, bins=100, color="#7bccc4", edgecolor="white", alpha=0.8)
    axes[1].axvline(0.95, color="red", linestyle="--", label="Suspicious (>0.95)")
    axes[1].axvline(0.90, color="orange", linestyle="--", label="High (>0.90)")
    axes[1].set_title(f"CLIP Cosine Similarity (val vs train)\nmax={clip_flat.max():.4f}")
    axes[1].set_xlabel("Cosine similarity")
    axes[1].legend()

plt.tight_layout()
plt.show()

for thresh in [0.85, 0.90, 0.95]:
    resnet_count = (resnet_flat >= thresh).sum()
    msg = f"ResNet: {resnet_count} pairs above {thresh}"
    if CLIP_AVAILABLE:
        clip_count = (clip_flat >= thresh).sum()
        msg += f" | CLIP: {clip_count} pairs above {thresh}"
    print(msg)

[Plot rendered locally — see executed notebook for images]


ResNet: 303 pairs above 0.85 | CLIP: 99393 pairs above 0.85
ResNet: 51 pairs above 0.9 | CLIP: 32175 pairs above 0.9
ResNet: 11 pairs above 0.95 | CLIP: 1037 pairs above 0.95


---
## 6. Inspect Most Similar Pairs

In [54]:
# Reload sample paths from cache if needed
if 'train_selected' not in dir() or 'val_selected' not in dir():
    import pickle
    with open(CACHE_DIR / 'selected_sample_ids.pkl', 'rb') as f:
        ids_data = pickle.load(f)
    _train_paths_raw = ids_data['train_paths']
    _val_paths_raw = ids_data['val_paths']
else:
    _train_paths_raw = [str(p.image_path) for p in train_selected]
    _val_paths_raw = [str(p.image_path) for p in val_selected]


def normalize_path(p):
    """Convert absolute Colab/Drive paths to local paths.
    Extracts everything from 'data/raw/...' onwards and resolves
    relative to the current PROJECT_ROOT."""
    p = str(p)
    marker = 'data/raw/'
    idx = p.find(marker)
    if idx >= 0:
        relative = p[idx:]
        local = PROJECT_ROOT / relative
        if local.exists():
            return str(local)
    # Fallback: return as-is if it exists, otherwise try just the filename
    if Path(p).exists():
        return p
    from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
    fallback = DEEPGLOBE_DATASET_DIR / 'train' / Path(p).name
    return str(fallback)


_train_paths = [normalize_path(p) for p in _train_paths_raw]
_val_paths = [normalize_path(p) for p in _val_paths_raw]

# Verify a sample path exists
print(f'Sample path: {_val_paths[0]}')
print(f'Exists: {Path(_val_paths[0]).exists()}')


def show_top_pairs(sim_matrix, val_paths, train_paths, method_name, n=10):
    flat_idx = np.argsort(sim_matrix.ravel())[-n:][::-1]
    val_idx = flat_idx // sim_matrix.shape[1]
    train_idx = flat_idx % sim_matrix.shape[1]

    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        vi, ti = val_idx[i], train_idx[i]
        sim_score = sim_matrix[vi, ti]

        val_img = np.array(Image.open(val_paths[vi]).convert('RGB'))
        train_img = np.array(Image.open(train_paths[ti]).convert('RGB'))

        axes[i, 0].imshow(val_img)
        axes[i, 0].set_title(f'VAL: {Path(val_paths[vi]).stem[-15:]}\nsim={sim_score:.4f}', fontsize=9,
                                   color='red' if sim_score >= 0.95 else 'black')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(train_img)
        axes[i, 1].set_title(f'TRAIN: {Path(train_paths[ti]).stem[-15:]}', fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 0].set_ylabel(f'sim={sim_score:.3f}', fontsize=12, fontweight='bold',
                                    color='red' if sim_score >= 0.95 else 'orange' if sim_score >= 0.90 else 'black')

    fig.suptitle(f'Top {n} Most Similar Val-Train Pairs ({method_name})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


show_top_pairs(resnet_sim, _val_paths, _train_paths, 'ResNet', n=20)

if CLIP_AVAILABLE:
    show_top_pairs(clip_sim, _val_paths, _train_paths, 'CLIP', n=20)


Sample path: /path/to/project/data/raw/deepglobe-road-extraction-dataset/train/630417_sat.jpg
Exists: True


[Plot rendered locally — see executed notebook for images]


[Plot rendered locally — see executed notebook for images]


---
## 8. Verdict

In [ ]:
print("Leakage Assessment")
print("=" * 60)

best_sim = resnet_sim
method = "ResNet"
if CLIP_AVAILABLE and clip_sim.max() > resnet_sim.max():
    best_sim = clip_sim
    method = "CLIP"

max_sim = best_sim.max()
n_above_90 = (best_sim.ravel() >= 0.90).sum()
n_above_95 = (best_sim.ravel() >= 0.95).sum()
total_pairs = best_sim.size

print(f"Method used: {method}")
print(f"Max similarity found: {max_sim:.4f}")
print(f"Pairs above 0.90: {n_above_90}/{total_pairs} ({n_above_90/total_pairs*100:.2f}%)")
print(f"Pairs above 0.95: {n_above_95}/{total_pairs} ({n_above_95/total_pairs*100:.2f}%)")

print()
if max_sim >= 0.95:
    print("HIGH LEAKAGE RISK: Near-duplicate tiles found between train and val.")
    print("Validation metrics are likely inflated. Consider:")
    print("  1. Removing the flagged val samples and re-evaluating")
    print("  2. Reporting both inflated and cleaned metrics in the write-up")
elif max_sim >= 0.90:
    print("MODERATE LEAKAGE RISK: Some highly similar (possibly adjacent) tiles found.")
    print("Metrics may be slightly optimistic. Acknowledge in the write-up.")
elif max_sim >= 0.80:
    print("LOW LEAKAGE RISK: Some similar scenes but no near-duplicates.")
    print("This is expected for satellite imagery from similar regions.")
else:
    print("MINIMAL LEAKAGE RISK: Train and val samples appear visually distinct.")
    print("The random split is reasonably safe for this dataset.")

---
## 9. Cleaned IoU — Combined Leakage Removal

Both methods found leaky samples at different thresholds:
- **ResNet** catches visual near-duplicates at threshold **0.93**
- **CLIP** catches semantic duplicates (same area, different appearance) at threshold **0.96**

We combine both — remove any val sample flagged by **either** method, then recompute IoU.
This uses only cached data (no inference needed).

In [55]:
import pickle
import pandas as pd

# Load cached data
with open(CACHE_DIR / 'top_k_selection.pkl', 'rb') as f:
    sel = pickle.load(f)

val_ious_all = sel['val_ious']
val_top_idx = sel['val_top_idx']

sim_data = np.load(CACHE_DIR / 'similarity_matrices.npz')
resnet_sim = sim_data['resnet_sim']
clip_sim = sim_data['clip_sim'] if 'clip_sim' in sim_data.files else None

# --- Thresholds (tuned from visual inspection in Section 6) ---
RESNET_THRESH = 0.92
CLIP_THRESH = 0.97

# Find leaky samples per method
resnet_max = resnet_sim.max(axis=1)
resnet_leaky = set(val_top_idx[resnet_max >= RESNET_THRESH])
print(f'ResNet >= {RESNET_THRESH}: {len(resnet_leaky)} leaky val samples')

if clip_sim is not None:
    clip_max = clip_sim.max(axis=1)
    clip_leaky = set(val_top_idx[clip_max >= CLIP_THRESH])
    print(f'CLIP >= {CLIP_THRESH}:   {len(clip_leaky)} leaky val samples')
else:
    clip_leaky = set()

# Combine: union of both methods
combined_leaky = resnet_leaky | clip_leaky
overlap = resnet_leaky & clip_leaky

print(f'\nUnion (either):       {len(combined_leaky)} unique leaky samples')
print(f'Intersection (both):  {len(overlap)} samples flagged by both methods')
print(f'ResNet only:          {len(resnet_leaky - clip_leaky)}')
print(f'CLIP only:            {len(clip_leaky - resnet_leaky)}')

ResNet >= 0.92: 21 leaky val samples
CLIP >= 0.97:   38 leaky val samples

Union (either):       48 unique leaky samples
Intersection (both):  11 samples flagged by both methods
ResNet only:          10
CLIP only:            27


In [56]:
# Sweep thresholds to show how each method behaves
thresholds = np.arange(0.80, 1.00, 0.01)

resnet_counts = []
clip_counts = []
combined_counts = []
cleaned_ious_resnet = []
cleaned_ious_clip = []

resnet_max_per_val = resnet_sim.max(axis=1)
clip_max_per_val = clip_sim.max(axis=1) if clip_sim is not None else None

for t in thresholds:
    r_leaky = set(val_top_idx[resnet_max_per_val >= t])
    resnet_counts.append(len(r_leaky))
    clean = [iou for i, iou in enumerate(val_ious_all) if i not in r_leaky]
    cleaned_ious_resnet.append(np.mean(clean) if clean else 0)

    if clip_max_per_val is not None:
        c_leaky = set(val_top_idx[clip_max_per_val >= t])
        clip_counts.append(len(c_leaky))
        combined = r_leaky | c_leaky
        combined_counts.append(len(combined))
        clean_c = [iou for i, iou in enumerate(val_ious_all) if i not in c_leaky]
        cleaned_ious_clip.append(np.mean(clean_c) if clean_c else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Number of flagged samples vs threshold
axes[0].plot(thresholds, resnet_counts, 'o-', label='ResNet', color='#2b8cbe', linewidth=2, markersize=4)
if clip_counts:
    axes[0].plot(thresholds, clip_counts, 's-', label='CLIP', color='#7bccc4', linewidth=2, markersize=4)
    axes[0].plot(thresholds, combined_counts, '^-', label='Combined (union)', color='#ef6548', linewidth=2, markersize=4)
axes[0].axvline(RESNET_THRESH, color='#2b8cbe', linestyle=':', alpha=0.5, label=f'ResNet cutoff ({RESNET_THRESH})')
if clip_sim is not None:
    axes[0].axvline(CLIP_THRESH, color='#7bccc4', linestyle=':', alpha=0.5, label=f'CLIP cutoff ({CLIP_THRESH})')
axes[0].set_xlabel('Similarity Threshold')
axes[0].set_ylabel('Number of Leaky Val Samples')
axes[0].set_title('Leaky Samples Detected vs Threshold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Cleaned IoU vs threshold
axes[1].plot(thresholds, cleaned_ious_resnet, 'o-', label='After ResNet removal', color='#2b8cbe', linewidth=2, markersize=4)
if cleaned_ious_clip:
    axes[1].plot(thresholds, cleaned_ious_clip, 's-', label='After CLIP removal', color='#7bccc4', linewidth=2, markersize=4)
axes[1].axhline(val_ious_all.mean(), color='red', linestyle='--', label=f'Original IoU: {val_ious_all.mean():.4f}')
axes[1].axvline(RESNET_THRESH, color='#2b8cbe', linestyle=':', alpha=0.5)
if clip_sim is not None:
    axes[1].axvline(CLIP_THRESH, color='#7bccc4', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Similarity Threshold')
axes[1].set_ylabel('Cleaned Mean IoU')
axes[1].set_title('Cleaned IoU vs Threshold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


[Plot rendered locally — see executed notebook for images]


In [57]:
# Show the leaky samples with their IoU scores
leaky_data = []
for idx in sorted(combined_leaky):
    source = []
    if idx in resnet_leaky:
        source.append('ResNet')
    if idx in clip_leaky:
        source.append('CLIP')
    leaky_data.append({
        'val_index': idx,
        'iou': val_ious_all[idx],
        'flagged_by': ' + '.join(source),
    })

leaky_df = pd.DataFrame(leaky_data).sort_values('iou', ascending=False)
print(f'Leaky samples IoU distribution:')
print(f'  Mean:   {leaky_df["iou"].mean():.4f}')
print(f'  Median: {leaky_df["iou"].median():.4f}')
print(f'  Min:    {leaky_df["iou"].min():.4f}')
print(f'  Max:    {leaky_df["iou"].max():.4f}')
print(f'\nTop 20 leaky samples by IoU:')
display(leaky_df.head(20).round(4))

Leaky samples IoU distribution:
  Mean:   0.8395
  Median: 0.8577
  Min:    0.7112
  Max:    0.9265

Top 20 leaky samples by IoU:


,val_index,iou,flagged_by
23,405,0.9265,CLIP
3,89,0.9183,ResNet + CLIP
1,60,0.9132,CLIP
47,926,0.9123,ResNet + CLIP
20,369,0.9095,ResNet
33,725,0.9039,ResNet
5,112,0.8973,ResNet + CLIP
4,99,0.8928,ResNet + CLIP
35,733,0.8907,ResNet + CLIP
9,210,0.8906,ResNet


In [58]:
# Visualize leaky samples alongside their most similar train image
if '_val_paths' not in dir():
    with open(CACHE_DIR / 'selected_sample_ids.pkl', 'rb') as f:
        ids_data = pickle.load(f)
    _val_paths_raw = ids_data['val_paths']
    _train_paths_raw = ids_data['train_paths']
    _val_paths = [normalize_path(p) for p in _val_paths_raw]
    _train_paths = [normalize_path(p) for p in _train_paths_raw]

resnet_max_scores = resnet_sim.max(axis=1)
clip_max_scores = clip_sim.max(axis=1) if clip_sim is not None else np.zeros_like(resnet_max_scores)
combined_score = np.maximum(resnet_max_scores, clip_max_scores)

leaky_in_topk = [i for i in range(len(val_top_idx)) if val_top_idx[i] in combined_leaky]
leaky_in_topk_sorted = sorted(leaky_in_topk, key=lambda i: combined_score[i], reverse=True)

N_SHOW = min(8, len(leaky_in_topk_sorted))
fig, axes = plt.subplots(N_SHOW, 2, figsize=(10, 3.5 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, vi in enumerate(leaky_in_topk_sorted[:N_SHOW]):
    best_method = 'ResNet'
    best_ti = resnet_sim[vi].argmax()
    best_score = resnet_sim[vi, best_ti]
    if clip_sim is not None and clip_sim[vi].max() > best_score:
        best_ti = clip_sim[vi].argmax()
        best_score = clip_sim[vi, best_ti]
        best_method = 'CLIP'

    val_img = np.array(Image.open(_val_paths[vi]).convert('RGB'))
    train_img = np.array(Image.open(_train_paths[best_ti]).convert('RGB'))

    iou_val = val_ious_all[val_top_idx[vi]]

    axes[row, 0].imshow(val_img)
    axes[row, 0].set_title(f'VAL (IoU={iou_val:.3f})', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(train_img)
    axes[row, 1].set_title(f'TRAIN (most similar)', fontsize=9)
    axes[row, 1].axis('off')

    color = 'red' if best_score >= 0.95 else 'orange'
    axes[row, 0].set_ylabel(f'{best_method}={best_score:.3f}', fontsize=11, fontweight='bold', color=color)

fig.suptitle('Leaky Samples: Validation Image vs Most Similar Training Image',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


[Plot rendered locally — see executed notebook for images]


In [59]:
# Compute cleaned IoU
clean_ious = [iou for i, iou in enumerate(val_ious_all) if i not in combined_leaky]
removed_ious = [val_ious_all[i] for i in combined_leaky]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: IoU distribution — leaky vs clean
axes[0].hist(clean_ious, bins=40, alpha=0.7, color='#41ae76',
             label=f'Clean (n={len(clean_ious)})', edgecolor='white')
axes[0].hist(removed_ious, bins=20, alpha=0.7, color='#ef6548',
             label=f'Leaky (n={len(removed_ious)})', edgecolor='white')
axes[0].axvline(np.mean(clean_ious), color='#2d8c4e', linestyle='--', linewidth=2,
               label=f'Clean mean: {np.mean(clean_ious):.4f}')
axes[0].axvline(np.mean(removed_ious), color='#c0392b', linestyle='--', linewidth=2,
               label=f'Leaky mean: {np.mean(removed_ious):.4f}')
axes[0].set_xlabel('Per-Sample IoU')
axes[0].set_ylabel('Count')
axes[0].set_title('IoU Distribution: Clean vs Leaky Samples')
axes[0].legend(fontsize=9)

# Plot 2: Before vs After IoU comparison (no count — separate scales)
labels = ['Mean IoU', 'Median IoU']
original = [val_ious_all.mean(), np.median(val_ious_all)]
cleaned = [np.mean(clean_ious), np.median(clean_ious)]
leaky = [np.mean(removed_ious), np.median(removed_ious)]

x = np.arange(len(labels))
w = 0.25
axes[1].bar(x - w, original, w, label=f'Original (n={len(val_ious_all)})', color='#2b8cbe')
axes[1].bar(x, cleaned, w, label=f'Cleaned (n={len(clean_ious)})', color='#41ae76')
axes[1].bar(x + w, leaky, w, label=f'Leaky (n={len(removed_ious)})', color='#ef6548')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('IoU')
axes[1].set_title('IoU Comparison: Original vs Cleaned vs Leaky')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.0)
for i in range(len(labels)):
    axes[1].text(i - w, original[i] + 0.02, f'{original[i]:.4f}', ha='center', fontsize=8)
    axes[1].text(i, cleaned[i] + 0.02, f'{cleaned[i]:.4f}', ha='center', fontsize=8)
    axes[1].text(i + w, leaky[i] + 0.02, f'{leaky[i]:.4f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()


[Plot rendered locally — see executed notebook for images]


In [60]:
inflation = val_ious_all.mean() - np.mean(clean_ious)
inflation_pct = inflation / val_ious_all.mean() * 100

print('Combined Leakage Impact')
print('=' * 60)
print(f'Method:              ResNet >= {RESNET_THRESH} OR CLIP >= {CLIP_THRESH}')
print(f'Total val samples:   {len(val_ious_all)}')
print(f'Leaky samples:       {len(combined_leaky)} ({len(combined_leaky)/len(val_ious_all)*100:.1f}%)')
print(f'Clean samples:       {len(clean_ious)}')
print()
print(f'Original mean IoU:   {val_ious_all.mean():.4f}')
print(f'Cleaned mean IoU:    {np.mean(clean_ious):.4f}')
print(f'Leaky samples IoU:   {np.mean(removed_ious):.4f}')
print(f'IoU inflation:       {inflation:.4f} ({inflation_pct:.1f}%)')
print()

if inflation_pct < 1:
    print('Conclusion: Leakage has NEGLIGIBLE impact (<1%). Metrics are reliable.')
elif inflation_pct < 3:
    print('Conclusion: Leakage has MINOR impact. Report both metrics:')
    print(f'  Reported IoU: {val_ious_all.mean():.4f}')
    print(f'  Cleaned IoU:  {np.mean(clean_ious):.4f} (conservative estimate)')
else:
    print('Conclusion: Leakage has SIGNIFICANT impact.')
    print(f'  The cleaned IoU ({np.mean(clean_ious):.4f}) is more trustworthy.')
    print('  Consider geographic splitting if tile coordinates are available.')

Combined Leakage Impact
Method:              ResNet >= 0.92 OR CLIP >= 0.97
Total val samples:   934
Leaky samples:       48 (5.1%)
Clean samples:       886

Original mean IoU:   0.6940
Cleaned mean IoU:    0.6862
Leaky samples IoU:   0.8395
IoU inflation:       0.0079 (1.1%)

Conclusion: Leakage has MINOR impact. Report both metrics:
  Reported IoU: 0.6940
  Cleaned IoU:  0.6862 (conservative estimate)
